# UR5e MuJoCo 시뮬레이션과 teleop

`ri_motion_v5_py312` 커널을 선택하세요. 마지막 셀에서 관절 슬라이더, Quest 3, SpaceMouse 중 하나로 OnRobot RG2가 장착된 UR5e를 조작합니다.

In [ ]:
from pathlib import Path
import subprocess
import sys

cwd = Path.cwd().resolve()
for repo_root in (cwd, *cwd.parents):
    if (repo_root / 'vai_robotics' / 'ur5e' / 'sim.py').is_file():
        break
else:
    raise FileNotFoundError('VAI_AIWORKER 저장소 안에서 노트북을 열어 주세요.')

simulator = [sys.executable, '-m', 'vai_robotics.ur5e.sim']

### 모델과 관절 동작 확인

뷰어를 열기 전에 UR5e 모델을 불러와 작은 관절 명령을 적용하고 결과를 확인합니다.

In [ ]:
check = subprocess.run([*simulator, '--check'], cwd=repo_root, capture_output=True, text=True)
print(check.stdout, end='')
if check.stderr:
    print(check.stderr, end='', file=sys.stderr)
check.check_returncode()

### 조작 방식 선택

`DEVICE`를 `sliders`, `quest`, `spacemouse` 중 하나로 설정하세요. Quest를 쓸 때는 `QUEST_HOST`에 헤드셋에서 접속할 PC의 LAN 주소를 입력하세요. SpaceMouse는 USB 수신기나 케이블로 PC에 연결합니다. 장치별 설정은 [UR5e 실행 안내](../README.md)를 참고하세요.

Quest와 SpaceMouse 모드에서는 **UR5e 조작 패널**이 함께 열립니다. 미리보기로 시작하려면 `START_IN_DRY_RUN = True`로 설정하세요. **DRY RUN**에서 클러치를 누른 채 목표를 입력하면 가상 팔이 따라갑니다. 놓으면 가상 팔이 현재 시뮬레이션 로봇의 자세로 돌아가고, 다시 누를 때도 그 자세에서 시작합니다. **SIM RUN**에서는 시뮬레이션 로봇을 직접 조작합니다. Quest는 선택한 손의 grip을 사용합니다. SpaceMouse는 `BUTTON`으로 선택한 버튼(기본 1번)을 한 번 눌렀다 놓은 뒤 다시 눌러 조작하세요. 패널에 선택한 버튼의 누름·놓음 상태가 표시됩니다.

SpaceMouse의 **0번과 1번을 함께 누르면 RG2가 닫힙니다**. 가상 그리퍼는 `GRIPPER_FORCE_N`과 현재 개도에 해당하는 제조사 속도 곡선을 따릅니다. 40 N에서는 110 mm부터 완전히 닫는 데 약 1.1–1.2초, 50 mm부터 약 0.4초가 걸립니다. 하나를 놓으면 **DRY RUN**은 보존된 SIM RUN 개도로 즉시 돌아갑니다. **SIM RUN**은 현재 개도에서 패널의 **열림 목표 폭**(기본 75 mm)까지 움직입니다. 값을 바꾸면 다음 열기부터 적용됩니다. 델타 모드에서는 두 버튼을 누른 채 캡을 움직여 팔도 이동할 수 있습니다.

`eef-delta`는 입력 장치의 이동·회전을 TCP 자세 변화량으로 사용합니다. `joint-delta`에서는 이동 세 축이 J1–J3, 회전 세 축이 J4–J6의 각도를 바꿉니다. 델타 모드에서는 패널에 변화량을 입력해 한 번 적용할 수도 있습니다.

`eef-absolute`와 `joint-absolute`에서는 **실측값 불러오기**로 시작해 목표값을 편집하세요. TCP는 베이스 기준 위치(mm)와 회전벡터(°), 관절은 J1–J6 각도(°)를 입력합니다. 클러치를 누른 채 **목표 적용**을 누르면 입력한 목표를 추종합니다. 클러치를 놓으면 추종을 끝내고, 다시 누르면 그때의 실측 자세에서 시작합니다.

`CONTROL_LAW`는 시작 제어 방식입니다. 실행 중에는 클러치와 그리퍼 버튼을 놓고 패널의 **목표 제어 → 제어 방식**에서 위치 제어와 임피던스 제어를 전환할 수 있습니다. **SIM RUN**에서 EEF 모드의 임피던스는 TCP 자리와 방향의 오차, 관절 모드의 임피던스는 각 관절의 각도 오차에 따라 반력을 냅니다. 임피던스 제어를 선택하면 패널에서 TCP 또는 J1–J6의 축별 강성과 감쇠를 편집할 수 있습니다. 클러치를 놓고 적용한 뒤 **SIM RUN**에서 다시 눌러 응답을 비교하세요. **DRY RUN**은 목표 자세 미리보기입니다.

In [ ]:
import signal

DEVICE = 'sliders'
QUEST_HOST = ''
HAND = 'right'
BUTTON = 1
COMMAND_MODE = 'eef-delta'  # eef-delta, eef-absolute, joint-delta, joint-absolute
CONTROL_LAW = 'position'  # position, impedance
GRIPPER_FORCE_N = 40.0
START_IN_DRY_RUN = False

if DEVICE == 'sliders':
    command = simulator
else:
    command = [sys.executable, '-u', '-m', 'vai_robotics.ur5e.teleop',
               '--backend', 'sim', '--input', DEVICE]
    command += ['--command-mode', COMMAND_MODE, '--control-law', CONTROL_LAW]
    command += ['--gripper-force-n', str(GRIPPER_FORCE_N)]
    if START_IN_DRY_RUN:
        command.append('--dry-run')
    if DEVICE == 'quest':
        if not QUEST_HOST:
            raise ValueError('QUEST_HOST에 PC의 LAN 주소를 입력하세요.')
        command += ['--quest-host', QUEST_HOST, '--hand', HAND]
    elif DEVICE == 'spacemouse':
        command += ['--button', str(BUTTON)]

with subprocess.Popen(command, cwd=repo_root, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True) as process:
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError(f'시뮬레이터 종료 코드 {process.returncode}')
    except KeyboardInterrupt:
        if process.poll() is None:
            process.send_signal(signal.SIGINT)
            process.wait()